In [ ]:
import os
import tarfile

def compress_parquet_folder(input_folder, output_file):
    """
    Compress all parquet files in a folder into a tar.gz archive.
    """
    with tarfile.open(output_file, "w:gz") as tar:
        for root, dirs, files in os.walk(input_folder):
            for file in files:
                if file.endswith(".parquet"):
                    full_path = os.path.join(root, file)
                    arcname = os.path.relpath(full_path, input_folder)
                    
                    print(f"Adding: {arcname}")
                    tar.add(full_path, arcname=arcname)

    print(f"\nDone! Archive created: {output_file}")



compress_parquet_folder("trades", "trades.tar.gz")
compress_parquet_folder("open_interest", "open_interest.tar.gz")
compress_parquet_folder("liquidations", "liquidations.tar.gz")
compress_parquet_folder("orderbook", "trades.tar.gz")

In [1]:
import subprocess
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "cryptohftdata"])

  Attempting uninstall: coverage
    Found existing installation: coverage 7.9.2
    Uninstalling coverage-7.9.2:
      Successfully uninstalled coverage-7.9.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [cryptohftdata]



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python3.12 -m pip install --upgrade pip


CompletedProcess(args=['/usr/bin/python3.12', '-m', 'pip', 'install', 'cryptohftdata'], returncode=0)

In [2]:
import pandas as pd
import cryptohftdata as chd
from datetime import datetime, timedelta
import os
import gc  # <--- Important for memory management

# --- CONFIGURATION ---
API_KEY = "467ab24e39287d1280d30cf4b3cd803217ac2b73bac1e71201251117c7cad952"
START_DATE = "2025-08-01"
END_DATE = "2025-08-30" 
SYMBOL = "BTCUSDT"
EXCHANGE = chd.exchanges.BINANCE_FUTURES

DROP_HEADERS = ['some_unused_column', 'internal_id', 'order_count', 'transaction_time', 'event_time', 'timestamp', 'first_update_id', 'final_update_id', 'prev_final_update_id', 'last_update_id', 'event_type] 

client = chd.CryptoHFTDataClient(api_key=API_KEY)

def optimize_floats(df):
    """Downcast floats to save 50% memory."""
    floats = df.select_dtypes(include=['float64']).columns
    df[floats] = df[floats].astype('float32')
    return df

def get_date_list(start, end):
    start_dt = datetime.strptime(start, "%Y-%m-%d")
    end_dt = datetime.strptime(end, "%Y-%m-%d")
    delta = end_dt - start_dt
    return [(start_dt + timedelta(days=i)).strftime("%Y-%m-%d") for i in range(delta.days + 1)]

def download_and_store():
    dates = get_date_list(START_DATE, END_DATE)
    categories = [
        ("trades", client.get_trades),
        ("open_interest", client.get_open_interest),
        ("liquidations", client.get_liquidations),
        ("orderbook", client.get_orderbook)
    ]

    for cat_name, fetch_func in categories:
        print(f"\n>>> Starting Category: {cat_name.upper()}")
        os.makedirs(cat_name, exist_ok=True)

        for date_str in dates:
            file_path = f"{cat_name}/{SYMBOL}_{date_str}.parquet"
            
            if os.path.exists(file_path):
                print(f"  [SKIP] {date_str} already exists.")
                continue

            try:
                print(f"  [FETCH] {date_str}...", end="\r")
                df = fetch_func(
                    symbol=SYMBOL,
                    exchange=EXCHANGE,
                    start_date=date_str,
                    end_date=date_str
                )

                if df is not None and not df.empty:
                    # 1. Drop unused columns immediately
                    df.drop(columns=[c for c in DROP_HEADERS if c in df.columns], errors='ignore', inplace=True)

                    # 2. Convert time
                    if 'received_time' in df.columns:
                        df['received_time'] = pd.to_datetime(df['received_time'], unit='ns')

                    # 3. Optimize Memory usage (Shrink floats)
                    df = optimize_floats(df)

                    # 4. Save to Disk
                    df.to_parquet(file_path, engine='pyarrow', compression='zstd', compression_level=9, index=False)
                    
                    print(f"  [SAVED] {date_str} | Rows: {len(df):,}")
                    
                    # 5. AGGRESSIVE MEMORY CLEANUP
                    del df
                    gc.collect() # Manually trigger garbage collection
                else:
                    print(f"  [EMPTY] {date_str} - No data found.")

            except Exception as e:
                print(f"\n  [ERROR] {date_str}: {e}")
                # Clean up even on error to prevent leaks
                if 'df' in locals(): del df
                gc.collect()

if __name__ == "__main__":
    download_and_store()
    print("\nDownload process complete.")

<>:35: SyntaxWarning: 'tuple' object is not callable; perhaps you missed a comma?
<>:35: SyntaxWarning: 'tuple' object is not callable; perhaps you missed a comma?
/tmp/ipykernel_43/3968736602.py:35: SyntaxWarning: 'tuple' object is not callable; perhaps you missed a comma?
  ("liquidations", client.get_liquidations)


TypeError: 'tuple' object is not callable